# Leitura dos dados do Jira
### Leitura na camada Bronze

#### install

In [1]:
!pip install pandas
!pip install pyarrow
!pip install nbformat

#### imports

In [12]:
import os
import json
import pandas as pd

In [187]:
# import def write_db(df,path)
%run "../functions/nb_function_write_db.ipynb"

In [188]:
#import def read_db(path)
%run "../functions/nb_function_read_db.ipynb"

In [24]:
df = read_parquet("../../data/bronze/db_bronze.parquet")

type(df)

pandas.core.frame.DataFrame

In [82]:
display(df)

,issues,project.project_id,project.project_name,project.extracted_at
0,[{'assignee': [{'email': 'gionni.lucio@fasttra...,DT-ENG,Data Engineering Platform,2026-01-22T13:55:26Z


### Transformação ISSUES

In [81]:
display(df["issues"])

0    [{'assignee': [{'email': 'gionni.lucio@fasttra...
Name: issues, dtype: object

In [45]:
df_issues = pd.json_normalize(df["issues"][0])

display(df_issues)

,assignee,id,issue_type,priority,status,timestamps
0,"[{'email': 'gionni.lucio@fasttrack.com', 'id':...",JIRA-0001,Bug,Low,Open,"[{'created_at': '2025-08-02T14:55:05Z', 'resol..."
1,"[{'email': 'francinne@fasttrack.com', 'id': 'u...",JIRA-0002,Bug,High,Resolved,"[{'created_at': '2025-03-09T12:39:26Z', 'resol..."
2,"[{'email': 'adyna@fasttrack.com', 'id': 'u005'...",JIRA-0003,Story,High,Open,"[{'created_at': '2025-06-30T17:06:48Z', 'resol..."
3,"[{'email': 'francinne@fasttrack.com', 'id': 'u...",JIRA-0004,Story,High,Done,"[{'created_at': '2025-11-22T20:24:58Z', 'resol..."
4,"[{'email': 'francinne@fasttrack.com', 'id': 'u...",JIRA-0005,Task,Medium,Resolved,"[{'created_at': '2025-01-06T01:08:44Z', 'resol..."
...,...,...,...,...,...,...
995,"[{'email': 'francinne@fasttrack.com', 'id': 'u...",JIRA-0996,Task,Low,Done,"[{'created_at': '2026-02-30T25:61:00Z', 'resol..."
996,"[{'email': 'guilherme@fasttrack.com', 'id': 'u...",JIRA-0997,Task,Medium,Open,"[{'created_at': '2026-02-30T25:61:00Z', 'resol..."
997,"[{'email': 'matheus.malta@fasttrack.com', 'id'...",JIRA-0998,Task,Low,Resolved,"[{'created_at': '2026-02-30T25:61:00Z', 'resol..."
998,"[{'email': 'adyna@fasttrack.com', 'id': 'u005'...",JIRA-0999,Task,High,Done,"[{'created_at': '2026-02-30T25:61:00Z', 'resol..."


In [111]:
df_issues.columns

Index(['assignee', 'id', 'issue_type', 'priority', 'status', 'timestamps'], dtype='object')

### Transformação ASSIGNEE

In [113]:
display(df_issues["assignee"])

0      [{'email': 'gionni.lucio@fasttrack.com', 'id':...
1      [{'email': 'francinne@fasttrack.com', 'id': 'u...
2      [{'email': 'adyna@fasttrack.com', 'id': 'u005'...
3      [{'email': 'francinne@fasttrack.com', 'id': 'u...
4      [{'email': 'francinne@fasttrack.com', 'id': 'u...
                             ...                        
995    [{'email': 'francinne@fasttrack.com', 'id': 'u...
996    [{'email': 'guilherme@fasttrack.com', 'id': 'u...
997    [{'email': 'matheus.malta@fasttrack.com', 'id'...
998    [{'email': 'adyna@fasttrack.com', 'id': 'u005'...
999    [{'email': 'francinne@fasttrack.com', 'id': 'u...
Name: assignee, Length: 1000, dtype: object

In [86]:
type(df_issues["assignee"].explode()[0])

dict

In [116]:
display(pd.json_normalize(df_issues["assignee"].explode()).add_prefix("assignee_"))

,assignee_email,assignee_id,assignee_name
0,gionni.lucio@fasttrack.com,u002,Gionni Lucio
1,francinne@fasttrack.com,u004,Francinne
2,adyna@fasttrack.com,u005,Adyna
3,francinne@fasttrack.com,u004,Francinne
4,francinne@fasttrack.com,u004,Francinne
...,...,...,...
995,francinne@fasttrack.com,u004,Francinne
996,guilherme@fasttrack.com,u003,Guilherme Francisco
997,matheus.malta@fasttrack.com,u001,Matheus Malta
998,adyna@fasttrack.com,u005,Adyna


In [119]:
df_issues_assignee = df_issues.join(pd.json_normalize(df_issues["assignee"].explode()).add_prefix("assignee_"))

In [120]:
display(df_issues_assignee)

,assignee,id,issue_type,priority,status,timestamps,assignee_email,assignee_id,assignee_name
0,"[{'email': 'gionni.lucio@fasttrack.com', 'id':...",JIRA-0001,Bug,Low,Open,"[{'created_at': '2025-08-02T14:55:05Z', 'resol...",gionni.lucio@fasttrack.com,u002,Gionni Lucio
1,"[{'email': 'francinne@fasttrack.com', 'id': 'u...",JIRA-0002,Bug,High,Resolved,"[{'created_at': '2025-03-09T12:39:26Z', 'resol...",francinne@fasttrack.com,u004,Francinne
2,"[{'email': 'adyna@fasttrack.com', 'id': 'u005'...",JIRA-0003,Story,High,Open,"[{'created_at': '2025-06-30T17:06:48Z', 'resol...",adyna@fasttrack.com,u005,Adyna
3,"[{'email': 'francinne@fasttrack.com', 'id': 'u...",JIRA-0004,Story,High,Done,"[{'created_at': '2025-11-22T20:24:58Z', 'resol...",francinne@fasttrack.com,u004,Francinne
4,"[{'email': 'francinne@fasttrack.com', 'id': 'u...",JIRA-0005,Task,Medium,Resolved,"[{'created_at': '2025-01-06T01:08:44Z', 'resol...",francinne@fasttrack.com,u004,Francinne
...,...,...,...,...,...,...,...,...,...
995,"[{'email': 'francinne@fasttrack.com', 'id': 'u...",JIRA-0996,Task,Low,Done,"[{'created_at': '2026-02-30T25:61:00Z', 'resol...",francinne@fasttrack.com,u004,Francinne
996,"[{'email': 'guilherme@fasttrack.com', 'id': 'u...",JIRA-0997,Task,Medium,Open,"[{'created_at': '2026-02-30T25:61:00Z', 'resol...",guilherme@fasttrack.com,u003,Guilherme Francisco
997,"[{'email': 'matheus.malta@fasttrack.com', 'id'...",JIRA-0998,Task,Low,Resolved,"[{'created_at': '2026-02-30T25:61:00Z', 'resol...",matheus.malta@fasttrack.com,u001,Matheus Malta
998,"[{'email': 'adyna@fasttrack.com', 'id': 'u005'...",JIRA-0999,Task,High,Done,"[{'created_at': '2026-02-30T25:61:00Z', 'resol...",adyna@fasttrack.com,u005,Adyna


In [121]:
df_issues_assignee_drop = df_issues_assignee.drop(columns=["assignee"])

In [122]:
display(df_issues_assignee_drop)

,id,issue_type,priority,status,timestamps,assignee_email,assignee_id,assignee_name
0,JIRA-0001,Bug,Low,Open,"[{'created_at': '2025-08-02T14:55:05Z', 'resol...",gionni.lucio@fasttrack.com,u002,Gionni Lucio
1,JIRA-0002,Bug,High,Resolved,"[{'created_at': '2025-03-09T12:39:26Z', 'resol...",francinne@fasttrack.com,u004,Francinne
2,JIRA-0003,Story,High,Open,"[{'created_at': '2025-06-30T17:06:48Z', 'resol...",adyna@fasttrack.com,u005,Adyna
3,JIRA-0004,Story,High,Done,"[{'created_at': '2025-11-22T20:24:58Z', 'resol...",francinne@fasttrack.com,u004,Francinne
4,JIRA-0005,Task,Medium,Resolved,"[{'created_at': '2025-01-06T01:08:44Z', 'resol...",francinne@fasttrack.com,u004,Francinne
...,...,...,...,...,...,...,...,...
995,JIRA-0996,Task,Low,Done,"[{'created_at': '2026-02-30T25:61:00Z', 'resol...",francinne@fasttrack.com,u004,Francinne
996,JIRA-0997,Task,Medium,Open,"[{'created_at': '2026-02-30T25:61:00Z', 'resol...",guilherme@fasttrack.com,u003,Guilherme Francisco
997,JIRA-0998,Task,Low,Resolved,"[{'created_at': '2026-02-30T25:61:00Z', 'resol...",matheus.malta@fasttrack.com,u001,Matheus Malta
998,JIRA-0999,Task,High,Done,"[{'created_at': '2026-02-30T25:61:00Z', 'resol...",adyna@fasttrack.com,u005,Adyna


### Transformação TIMESTAMPS

In [128]:
display(pd.json_normalize(df_issues_assignee_drop["timestamps"].explode()).add_prefix("timestamps_"))

,timestamps_created_at,timestamps_resolved_at
0,2025-08-02T14:55:05Z,None
1,2025-03-09T12:39:26Z,2025-03-10T10:39:26Z
2,2025-06-30T17:06:48Z,None
3,2025-11-22T20:24:58Z,2025-11-23T07:24:58Z
4,2025-01-06T01:08:44Z,2025-01-07T09:08:44Z
...,...,...
995,2026-02-30T25:61:00Z,not_a_date
996,2026-02-30T25:61:00Z,not_a_date
997,2026-02-30T25:61:00Z,not_a_date
998,2026-02-30T25:61:00Z,not_a_date


In [129]:
df_issues_assignee_drop_timestamps = df_issues_assignee_drop.join(
                                        pd.json_normalize(
                                            df_issues_assignee_drop["timestamps"].
                                            explode()
                                        )
                                        .add_prefix("timestamps_")
                                    )

In [132]:
display(df_issues_assignee_drop_timestamps)

,id,issue_type,priority,status,timestamps,assignee_email,assignee_id,assignee_name,timestamps_created_at,timestamps_resolved_at
0,JIRA-0001,Bug,Low,Open,"[{'created_at': '2025-08-02T14:55:05Z', 'resol...",gionni.lucio@fasttrack.com,u002,Gionni Lucio,2025-08-02T14:55:05Z,None
1,JIRA-0002,Bug,High,Resolved,"[{'created_at': '2025-03-09T12:39:26Z', 'resol...",francinne@fasttrack.com,u004,Francinne,2025-03-09T12:39:26Z,2025-03-10T10:39:26Z
2,JIRA-0003,Story,High,Open,"[{'created_at': '2025-06-30T17:06:48Z', 'resol...",adyna@fasttrack.com,u005,Adyna,2025-06-30T17:06:48Z,None
3,JIRA-0004,Story,High,Done,"[{'created_at': '2025-11-22T20:24:58Z', 'resol...",francinne@fasttrack.com,u004,Francinne,2025-11-22T20:24:58Z,2025-11-23T07:24:58Z
4,JIRA-0005,Task,Medium,Resolved,"[{'created_at': '2025-01-06T01:08:44Z', 'resol...",francinne@fasttrack.com,u004,Francinne,2025-01-06T01:08:44Z,2025-01-07T09:08:44Z
...,...,...,...,...,...,...,...,...,...,...
995,JIRA-0996,Task,Low,Done,"[{'created_at': '2026-02-30T25:61:00Z', 'resol...",francinne@fasttrack.com,u004,Francinne,2026-02-30T25:61:00Z,not_a_date
996,JIRA-0997,Task,Medium,Open,"[{'created_at': '2026-02-30T25:61:00Z', 'resol...",guilherme@fasttrack.com,u003,Guilherme Francisco,2026-02-30T25:61:00Z,not_a_date
997,JIRA-0998,Task,Low,Resolved,"[{'created_at': '2026-02-30T25:61:00Z', 'resol...",matheus.malta@fasttrack.com,u001,Matheus Malta,2026-02-30T25:61:00Z,not_a_date
998,JIRA-0999,Task,High,Done,"[{'created_at': '2026-02-30T25:61:00Z', 'resol...",adyna@fasttrack.com,u005,Adyna,2026-02-30T25:61:00Z,not_a_date


In [134]:
df_issues_assignee_drop_timestamps_drop = df_issues_assignee_drop_timestamps.drop(columns=["timestamps"])

In [135]:
display(df_issues_assignee_drop_timestamps_drop)

,id,issue_type,priority,status,assignee_email,assignee_id,assignee_name,timestamps_created_at,timestamps_resolved_at
0,JIRA-0001,Bug,Low,Open,gionni.lucio@fasttrack.com,u002,Gionni Lucio,2025-08-02T14:55:05Z,None
1,JIRA-0002,Bug,High,Resolved,francinne@fasttrack.com,u004,Francinne,2025-03-09T12:39:26Z,2025-03-10T10:39:26Z
2,JIRA-0003,Story,High,Open,adyna@fasttrack.com,u005,Adyna,2025-06-30T17:06:48Z,None
3,JIRA-0004,Story,High,Done,francinne@fasttrack.com,u004,Francinne,2025-11-22T20:24:58Z,2025-11-23T07:24:58Z
4,JIRA-0005,Task,Medium,Resolved,francinne@fasttrack.com,u004,Francinne,2025-01-06T01:08:44Z,2025-01-07T09:08:44Z
...,...,...,...,...,...,...,...,...,...
995,JIRA-0996,Task,Low,Done,francinne@fasttrack.com,u004,Francinne,2026-02-30T25:61:00Z,not_a_date
996,JIRA-0997,Task,Medium,Open,guilherme@fasttrack.com,u003,Guilherme Francisco,2026-02-30T25:61:00Z,not_a_date
997,JIRA-0998,Task,Low,Resolved,matheus.malta@fasttrack.com,u001,Matheus Malta,2026-02-30T25:61:00Z,not_a_date
998,JIRA-0999,Task,High,Done,adyna@fasttrack.com,u005,Adyna,2026-02-30T25:61:00Z,not_a_date


### Analisa dados e faz limpeza

In [159]:
df = df_issues_assignee_drop_timestamps_drop.copy()

In [163]:
display(df["timestamps_created_at"])

0      2025-08-02T14:55:05Z
1      2025-03-09T12:39:26Z
2      2025-06-30T17:06:48Z
3      2025-11-22T20:24:58Z
4      2025-01-06T01:08:44Z
               ...         
995    2026-02-30T25:61:00Z
996    2026-02-30T25:61:00Z
997    2026-02-30T25:61:00Z
998    2026-02-30T25:61:00Z
999    2026-02-30T25:61:00Z
Name: timestamps_created_at, Length: 1000, dtype: object

In [166]:
# getting errors to fix
df.loc[
    pd.to_datetime(df["timestamps_created_at"], errors="coerce").isna(),
    "timestamps_created_at"
]

990    2026-02-30T25:61:00Z
991    2026-02-30T25:61:00Z
992    2026-02-30T25:61:00Z
993    2026-02-30T25:61:00Z
994    2026-02-30T25:61:00Z
995    2026-02-30T25:61:00Z
996    2026-02-30T25:61:00Z
997    2026-02-30T25:61:00Z
998    2026-02-30T25:61:00Z
999    2026-02-30T25:61:00Z
Name: timestamps_created_at, dtype: object

In [167]:
# fix errors with data (timestamps_created_at):
df["timestamps_created_at"] = pd.to_datetime(
    df["timestamps_created_at"],
    errors="coerce"
)

# checking:
df.loc[
    pd.to_datetime(df["timestamps_created_at"], errors="coerce").isna(),
    "timestamps_created_at"
]

990   NaT
991   NaT
992   NaT
993   NaT
994   NaT
995   NaT
996   NaT
997   NaT
998   NaT
999   NaT
Name: timestamps_created_at, dtype: datetime64[ns, UTC]

In [ ]:
# converting after fix values of february wrong dates

df["timestamps_created_at"] = pd.to_datetime(df["timestamps_created_at"])

In [170]:
# checking:
df.loc[
    pd.to_datetime(df["timestamps_resolved_at"], errors="coerce").isna(),
    "timestamps_resolved_at"
]

0            None
2            None
5            None
6            None
9            None
          ...    
995    not_a_date
996    not_a_date
997    not_a_date
998    not_a_date
999    not_a_date
Name: timestamps_resolved_at, Length: 196, dtype: object

In [172]:
# fixing errors with data (timestamps_resolved_at):
df["timestamps_resolved_at"] = pd.to_datetime(
    df["timestamps_resolved_at"],
    errors="coerce"
)

# checking again:
df.loc[
    pd.to_datetime(df["timestamps_resolved_at"], errors="coerce").isna(),
    "timestamps_resolved_at"
]

0     NaT
2     NaT
5     NaT
6     NaT
9     NaT
       ..
995   NaT
996   NaT
997   NaT
998   NaT
999   NaT
Name: timestamps_resolved_at, Length: 196, dtype: datetime64[ns, UTC]

In [176]:
df.dtypes

id                                     object
issue_type                             object
priority                               object
status                                 object
assignee_email                 string[python]
assignee_id                            object
assignee_name                          object
timestamps_created_at     datetime64[ns, UTC]
timestamps_resolved_at    datetime64[ns, UTC]
dtype: object

In [177]:
df["assignee_email"] = df["assignee_email"].astype("string")
df["id"] = df["id"].astype("string")
df["issue_type"] = df["issue_type"].astype("string")
df["priority"] = df["priority"].astype("string")
df["status"] = df["status"].astype("string")
df["assignee_email"] = df["assignee_email"].astype("string")
df["assignee_id"] = df["assignee_id"].astype("string")
df["assignee_name"] = df["assignee_name"].astype("string")

In [178]:
df.dtypes

id                             string[python]
issue_type                     string[python]
priority                       string[python]
status                         string[python]
assignee_email                 string[python]
assignee_id                    string[python]
assignee_name                  string[python]
timestamps_created_at     datetime64[ns, UTC]
timestamps_resolved_at    datetime64[ns, UTC]
dtype: object

#### Importa função write_db(df,caminho)

In [180]:
write_db(df_issues_assignee_drop_timestamps_drop,"../../data/silver/db_silver.parquet")

db_bronze.parquet gerado com sucesso!
